# Problem 2. 영화 리뷰 감성 분류하기

## 문제 개요
영화 정보 서비스는 사용자가 남긴 짧은 한 줄 리뷰가 **긍정(1)인지 부정(0)인지** 자동으로 분류하려 합니다.
리뷰 텍스트로 감성을 예측하는 모델을 만드세요.

## 파일 구조
```
problem02/
├── work.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 20,000개 (label 포함)
- `data/test.csv` : 10,000개 (label 없음)

| 컬럼 | 설명 |
|---|---|
| id | 고유 번호 (`R00000` 형식) |
| review | 리뷰 본문 (최대 140자) |
| **label** | **1 = 긍정, 0 = 부정 (예측 대상)** |

### 데이터 특징
- 긍정/부정 비율은 거의 1:1입니다.
- 맞춤법 오류, 띄어쓰기 생략, 초성(ㅋㅋ, ㅠㅠ), 반복 문자가 매우 많습니다.
- 출처: NSMC(Naver Sentiment Movie Corpus, CC0) 일부를 대회용으로 재구성

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`problem02/`)에 **`submission.csv`** 로 저장

## 평가
**Accuracy (정확도)** — 높을수록 좋습니다.

## 제출 형식
`submission.csv`

| id | label |
|---|---|
| R20000 | 1 |
| R20001 | 0 |

- 컬럼명 `id`, `label`, label은 0 또는 1 정수
- 포맷이 틀리면 0점 처리됩니다.

## 제출 검증 규칙
- 테스트의 모든 ID를 정확히 한 번씩 제출합니다. 누락·중복·추가 ID는 오류이며, 행 순서는 자유입니다(채점 시 ID로 정렬합니다).
- CSV는 UTF-8(또는 UTF-8 BOM), 쉼표 구분, 헤더 포함 형식입니다. 아래 두 컬럼만 제출하며 컬럼 순서는 자유입니다.
- 비어 있는 셀, NaN, 무한대, 깨진 CSV는 오류입니다. 올바르지 않은 제출은 해당 문제 0점과 ERROR로 기록합니다.
- 최종 파일은 이 문제 폴더의 `submission.csv`입니다. 다른 이름이나 하위 폴더 파일은 수집하지 않습니다.
- 컬럼: `id,label`; label은 0 또는 1 정수입니다. 확률 대신 최종 클래스를 제출하세요.
- 행 수: 10,000.
- id는 R20000~R29999의 문자열입니다.

## 100점 환산
`score = 100 × Accuracy` (모든 테스트 행에서 정확히 일치한 클래스의 비율). 세 문제 합계는 300점입니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print("train:", train.shape, "test:", test.shape)
train.head()

## TODO
데이터를 확인하고 전처리·모델·검증 방법을 구현하세요. 아래는 원문에서 제공한 간단한 베이스라인입니다. 필요에 맞게 수정하세요.

In [ ]:
import re, warnings; warnings.filterwarnings("ignore")
from scipy.sparse import hstack
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin

X, y, X_test = train.review.values, train.label.values, test.review.values
assert train.review.notna().all() and test.review.notna().all()

def norm(t):                                    # 반복 문자 정규화: ㅋㅋㅋㅋㅋ -> ㅋㅋ, 재밌어어어어 -> 재밌어어
    return re.sub(r"(.)\1{2,}", r"\1\1", str(t))
JAMO = re.compile("[ㄱ-ㅣ]")
def hand(t):                                    # 직접 설계한 표현 특징(길이, 부호, ㅋㅋ/ㅠㅠ, 반복, 숫자/평점, 문장 끝 형태)
    n = norm(t); runs = [len(m.group(0)) for m in re.finditer(r"(.)\1+", str(t))]
    return [len(n), len(n.split()), n.count("!"), n.count("?"), n.count(".") + n.count("…"), n.count("~"), n.count("ㅋ"), n.count("ㅎ"),
            n.count("ㅠ") + n.count("ㅜ"), len(JAMO.findall(n)), max(runs) if runs else 0, int(bool(re.search(r"\d", n))),
            int(bool(re.search(r"\d+\s*점|별\s*\d|\d/10", n))), int(n.strip().endswith(("?", "??"))), int(n.strip().endswith(("!", "!!", "~", "ㅋ", "ㅎ")))]

class LexFeatures(BaseEstimator, TransformerMixin):
    """train 에서 자주 나오는(min_df) 단어/표현(글자 1~3gram + 어절 1~2gram)의 긍정/부정 로그오즈를 추출하고, 문장별 집계 특징으로 변환"""
    def __init__(self, min_df=12, top=400, alpha=1.0):
        self.min_df = min_df; self.top = top; self.alpha = alpha
    def fit(self, T, y):
        T = [norm(t) for t in T]; y = np.asarray(y)
        self.cv1_ = CountVectorizer(analyzer="char_wb", ngram_range=(1, 3), binary=True, min_df=self.min_df).fit(T)
        self.cv2_ = CountVectorizer(analyzer="word", ngram_range=(1, 2), binary=True, token_pattern=r"\S+", min_df=self.min_df).fit(T)
        M = hstack([self.cv1_.transform(T), self.cv2_.transform(T)]).tocsr()
        p = self.alpha + M[y == 1].sum(0); q = self.alpha + M[y == 0].sum(0)
        self.lo_ = np.asarray(np.log((p / (y == 1).sum()) / (q / (y == 0).sum()))).ravel()     # 표현별 긍정/부정 로그오즈
        order = np.argsort(self.lo_); self.neg_idx_ = order[:self.top]; self.pos_idx_ = order[-self.top:]
        self.strong_ = np.abs(self.lo_) >= np.quantile(np.abs(self.lo_), 0.8)                  # 강한 극성 표현만
        names = np.concatenate([self.cv1_.get_feature_names_out(), self.cv2_.get_feature_names_out()])
        self.top_pos_ = names[self.pos_idx_[::-1][:15]].tolist(); self.top_neg_ = names[self.neg_idx_[:15]].tolist()
        return self
    def transform(self, T):
        Tn = [norm(t) for t in T]; M = hstack([self.cv1_.transform(Tn), self.cv2_.transform(Tn)]).tocsr()
        L = M.multiply(self.lo_).tocsr(); pos = M[:, self.pos_idx_].sum(1).A1; neg = M[:, self.neg_idx_].sum(1).A1
        s = M.multiply(self.strong_.astype(float)).tocsr(); Ls = s.multiply(self.lo_).tocsr()
        tot = L.sum(1).A1; cnt = np.maximum(M.sum(1).A1, 1)
        mx = np.array([r.max() if r.nnz else 0 for r in L]); mn = np.array([r.min() if r.nnz else 0 for r in L])
        H = np.array([hand(t) for t in T], dtype=float)
        return np.column_stack([pos, neg, pos - neg, tot, tot / cnt, mx, mn, Ls.sum(1).A1, s.sum(1).A1, H])

class LexHGB(BaseEstimator, ClassifierMixin):
    """추출한 표현 특징(24개)만으로 학습하는 HGB"""
    def fit(self, T, y):
        self.lex_ = LexFeatures().fit(T, y)
        self.m_ = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=15, random_state=0).fit(self.lex_.transform(T), y)
        self.classes_ = self.m_.classes_; return self
    def predict_proba(self, T): return self.m_.predict_proba(self.lex_.transform(T))

class NBLR(BaseEstimator, ClassifierMixin):
    """NB-SVM 계열: 이진 n-gram(글자 1~4, 어절 1~2)에 NB 로그비율 r 을 곱한 특징 + 로지스틱 회귀"""
    def __init__(self, C=8, alpha=1.0):
        self.C = C; self.alpha = alpha
    def fit(self, T, y):
        T = [norm(t) for t in T]
        self.v_ = [CountVectorizer(analyzer="char_wb", ngram_range=(1, 4), binary=True, min_df=2).fit(T),
                   CountVectorizer(analyzer="word", ngram_range=(1, 2), binary=True, token_pattern=r"\S+", min_df=2).fit(T)]
        M = hstack([v.transform(T) for v in self.v_]).tocsr(); p = self.alpha + M[y == 1].sum(0); q = self.alpha + M[y == 0].sum(0)
        self.r_ = np.asarray(np.log((p / p.sum()) / (q / q.sum()))).ravel()
        self.lr_ = LogisticRegression(C=self.C, max_iter=3000).fit(M.multiply(self.r_).tocsr(), y); self.classes_ = self.lr_.classes_; return self
    def predict_proba(self, T):
        T = [norm(t) for t in T]; M = hstack([v.transform(T) for v in self.v_]).tocsr()
        return self.lr_.predict_proba(M.multiply(self.r_).tocsr())

def build():
    return {
        "cnb":     make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True, min_df=2, preprocessor=norm), ComplementNB(alpha=0.5)),
        "mnb":     make_pipeline(CountVectorizer(analyzer="char_wb", ngram_range=(1, 4), binary=True, min_df=2, preprocessor=norm), MultinomialNB(alpha=0.5)),
        "nblr":    NBLR(C=8),
        "lex_hgb": LexHGB(),
    }

rank = lambda p: pd.Series(p).rank().values / len(p)        # 모델마다 확률 보정이 달라 순위 평균으로 결합
def to_label(score, pos_rate):                               # 상위 pos_rate 비율을 긍정(1)로 (문제: 긍정/부정 거의 1:1)
    return (score >= np.quantile(score, 1 - pos_rate)).astype(int)

# ---- 5-Fold CV ----
oof = {k: np.zeros(len(X)) for k in build()}
for a, b in StratifiedKFold(5, shuffle=True, random_state=42).split(X, y):
    for k, m in build().items():
        oof[k][b] = m.fit(X[a], y[a]).predict_proba(X[b])[:, 1]
for k in oof: print(f"{k:8s} CV acc: {((oof[k] >= .5) == y).mean():.4f}")
ens = np.mean([rank(p) for p in oof.values()], axis=0)
cv_acc = float((to_label(ens, y.mean()) == y).mean()); print(f"ensemble(rank avg) CV acc: {cv_acc:.4f} -> 환산 점수 약 {100 * cv_acc:.1f}   (이전 v1 NB 3종: 0.8478)")

# ---- 전체 train 재학습 후 test 예측 ----
models = build(); score = np.mean([rank(m.fit(X, y).predict_proba(X_test)[:, 1]) for m in models.values()], axis=0)
pred = to_label(score, y.mean())
lex = models["lex_hgb"].lex_; print("추출된 긍정 표현 상위:", lex.top_pos_[:10]); print("추출된 부정 표현 상위:", lex.top_neg_[:10])
print("test 예측 긍정 비율: %.3f (train %.3f)" % (pred.mean(), y.mean()))

In [ ]:
submission = pd.DataFrame({"id": test["id"], "label": pred.astype(int)})
submission.to_csv("submission.csv", index=False)

In [ ]:
submission = pd.read_csv("submission.csv")
print(submission.shape)
assert len(submission) == len(test)
assert submission["id"].is_unique
assert set(submission["id"]) == set(test["id"])
submission.head()
assert list(submission.columns) == ["id", "label"] and set(submission.label.unique()) <= {0, 1} and submission.label.dtype.kind == "i"